In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%PandasProfile
### cell 10 ###

# Compute per-player mean LOS difference more efficiently
los_diff = (
    df.groupby(['gameId', 'playId', 'nflId'], as_index=False, sort=False)
      .agg(
          max_diff=('los_diff', 'max'),
          min_diff=('los_diff', 'min'),
          posTeam=('posTeam', 'first')
      )
      .assign(
          los_diff=lambda d: d['max_diff'].where(d['posTeam'] == 1, d['min_diff'])
      )
      .loc[:, ['nflId', 'los_diff']]
      .groupby('nflId', as_index=False, sort=False)['los_diff']
      .mean()
)

# Rename snap columns and join on the reduced los_diff Series
snap_speed = (
    snap_speed
      .rename(columns={'s': 'snap_s', 'a': 'snap_a'})
      .join(
          los_diff.set_index('nflId')['los_diff']
                  .rename('snap_los_diff'),
          on='nflId',
          how='inner'
      )
)